# Step 2: Task-Agnostic θ_BS (KILLER EXPERIMENT)

**Claim**: θ_BS trained on Task A (channel estimation) transfers to Task B (beam prediction).

## Experiment Design
1. Train on Task A (channel estimation): learn E_A + θ_task_A + θ_BS
2. Freeze θ_BS
3. Train new θ_task_B for Task B (beam prediction), using frozen E_A and θ_BS
4. Compare vs. training Task B from scratch (no θ_BS)

If θ_BS helps Task B → it captures site-specific info independent of task → "site foundation representation"

In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.training.trainer import load_checkpoint

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
SAVE_PREFIX = 'phase2'
TEST_BS = [6, 7]

## Step 2-1: Task A → Task B Transfer Results

학습은 `phase2_train.py --step 2-1`로 수행. 여기서는 체크포인트 메타에서 학습 곡선 로드.

In [ ]:
# Load step 2-1 checkpoints: with_theta, without_theta, scratch
from src.experiments.2_task_agnostic.train import TaskBModel, PowerProfileHead
from src.models.estimator import SharedEncoder, create_model

for test_bs in TEST_BS:
    print(f'\n=== BS{test_bs} ===')

    # Build correct model architectures matching what phase2_train.py saved
    ref_model = create_model(site_integration='film', site_embed_dim=64)
    ckpt_configs = {
        'With θ_BS': (
            f'{SAVE_PREFIX}/2-1_with_theta_bs{test_bs}',
            lambda: TaskBModel(
                encoder=SharedEncoder(2, 64, 3, 3),
                site_embedding=ref_model.site_embedding.__class__(64),
                site_injection=ref_model.site_injection.__class__(64, 64),
                task_head=PowerProfileHead(),
            ),
        ),
        'Without θ_BS': (
            f'{SAVE_PREFIX}/2-1_without_theta_bs{test_bs}',
            lambda: TaskBModel(encoder=SharedEncoder(2, 64, 3, 3), task_head=PowerProfileHead()),
        ),
        'From scratch': (
            f'{SAVE_PREFIX}/2-1_scratch_bs{test_bs}',
            lambda: TaskBModel(encoder=SharedEncoder(2, 64, 3, 3), task_head=PowerProfileHead()),
        ),
    }

    metas = {}
    for label, (ckpt_name, model_fn) in ckpt_configs.items():
        try:
            m = model_fn().to(device)
            meta = load_checkpoint(m, ckpt_name, device=device)
            metas[label] = meta
            print(f'  {label}: best_val={meta["best_val"]:.6f} (epoch {meta["best_epoch"]})')
        except FileNotFoundError:
            print(f'  {label}: checkpoint not found')

    # Plot convergence
    if metas and all('val_losses' in m for m in metas.values()):
        fig, ax = plt.subplots(figsize=(8, 5))
        for label, meta in metas.items():
            ax.plot(meta['val_losses'], label=label, linewidth=2)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Normalized MSE')
        ax.set_title(f'Task B (Power Profile) — BS{test_bs}')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # Check criterion: with theta_BS should be 10%+ better
        if 'With θ_BS' in metas and 'Without θ_BS' in metas:
            with_val = metas['With θ_BS']['best_val']
            without_val = metas['Without θ_BS']['best_val']
            improvement = (without_val - with_val) / without_val * 100
            print(f'  θ_BS improvement: {improvement:.1f}% [{"PASS" if improvement > 10 else "CHECK"}]')
    elif metas:
        print('  val_losses not in checkpoint meta. Re-run phase2_train.py')

## Step 2-2: Pre-trained E as Downstream Backbone

학습은 `phase2_train.py --step 2-2`로 수행.

In [ ]:
# Load step 2-2 checkpoints: frozen pretrained E, frozen random E, full scratch
from src.models.baselines import PlainEstimator

for test_bs in TEST_BS:
    print(f'\n=== BS{test_bs} ===')
    ckpt_configs = {
        'Frozen pretrained E': f'{SAVE_PREFIX}/2-2_frozen_E_bs{test_bs}',
        'Frozen random E': f'{SAVE_PREFIX}/2-2_random_E_bs{test_bs}',
        'Full from scratch': f'{SAVE_PREFIX}/2-2_full_bs{test_bs}',
    }

    metas = {}
    for label, ckpt_name in ckpt_configs.items():
        dummy = PlainEstimator(encoder_channels=64, encoder_blocks=3).to(device)
        try:
            meta = load_checkpoint(dummy, ckpt_name, device=device)
            metas[label] = meta
            best_db = 10 * np.log10(meta['best_val'])
            print(f'  {label}: {best_db:.2f} dB (epoch {meta["best_epoch"]})')
        except FileNotFoundError:
            print(f'  {label}: checkpoint not found')

    if metas and all('val_losses' in m for m in metas.values()):
        fig, ax = plt.subplots(figsize=(8, 5))
        for label, meta in metas.items():
            ax.plot(10*np.log10(meta['val_losses']), label=label, linewidth=2)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('NMSE (dB)')
        ax.set_title(f'Pre-trained E as Backbone — BS{test_bs}')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # Check criterion: frozen pretrained > frozen random by 3+ dB
        if 'Frozen pretrained E' in metas and 'Frozen random E' in metas:
            pre_db = 10 * np.log10(metas['Frozen pretrained E']['best_val'])
            rand_db = 10 * np.log10(metas['Frozen random E']['best_val'])
            diff = rand_db - pre_db
            print(f'  Pretrained E advantage: {diff:.1f} dB [{"PASS" if diff > 3 else "CHECK"}]')